# 👁️ Diabetic Retinopathy 2-Stage Deep Learning Model Training Notebook

This notebook trains the 2-stage Diabetic Retinopathy detection and severity grading system using the APTOS 2019 Blindness Detection dataset on **Google Colab** (using free T4 GPU).

### Framework Architecture:
- **Stage 1 (Binary)**: ResNet18 classifier (Healthy vs. DR Present).
- **Stage 2 (Multi-Class)**: EfficientNetB0 classifier (Mild, Moderate, Severe, Proliferative DR).
- **Preprocessing**: OpenCV CLAHE enhancement + standard ImageNet normalization.

In [ ]:
# 1. Environment Setup & Requirements Installation
!pip install -q opencv-python pillow pandas numpy scikit-learn matplotlib seaborn albumentations
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Kaggle API Setup & Dataset Download
import os
from google.colab import files

print("Please upload your kaggle.json API key file...")
uploaded = files.upload()

if 'kaggle.json' in uploaded:
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle API key configured successfully!")
    
    # Download APTOS 2019 dataset
    !kaggle competitions download -c aptos2019-blindness-detection
    !mkdir -p dataset
    !unzip -q aptos2019-blindness-detection.zip -d dataset/
    print("APTOS 2019 dataset unzipped to ./dataset/")
else:
    print("Warning: kaggle.json not uploaded. Please ensure dataset CSV and images exist in ./dataset/")

In [ ]:
# 3. Import Project Modules
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from src.dataset import APTOSDataset
from src.models import build_stage1_model, build_stage2_model
from src.train import train_model_pipeline
from src.preprocessing import get_pytorch_transforms

# Load dataset CSV
df = pd.read_csv('./dataset/train.csv')
print(f"Total images: {len(df)}")
print(df['diagnosis'].value_counts().sort_index())

In [ ]:
# 4. Stratified Split (Train, Val, Test)
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['diagnosis'])
train_df, val_df = train_test_split(train_df, test_size=0.15, random_state=42, stratify=train_df['diagnosis'])

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

In [ ]:
# 5. Train Stage 1 Model (Binary: Healthy vs. DR Present)
img_dir = './dataset/train_images'

stage1_train_ds = APTOSDataset(train_df, img_dir, stage=1, transform=get_pytorch_transforms(224, is_train=True))
stage1_val_ds = APTOSDataset(val_df, img_dir, stage=1, transform=get_pytorch_transforms(224, is_train=False))

stage1_train_loader = DataLoader(stage1_train_ds, batch_size=32, shuffle=True, num_workers=2)
stage1_val_loader = DataLoader(stage1_val_ds, batch_size=32, shuffle=False, num_workers=2)

stage1_model = build_stage1_model(backbone='resnet18', pretrained=True)

print("=== Training Stage 1 Binary Model ===")
s1_history = train_model_pipeline(
    model=stage1_model,
    train_loader=stage1_train_loader,
    val_loader=stage1_val_loader,
    num_epochs=12,
    lr=1e-4,
    save_path='./models/stage1_binary.pth'
)

In [ ]:
# 6. Train Stage 2 Model (Multi-Class Severity: Mild, Moderate, Severe, Proliferative)
stage2_train_ds = APTOSDataset(train_df, img_dir, stage=2, transform=get_pytorch_transforms(224, is_train=True))
stage2_val_ds = APTOSDataset(val_df, img_dir, stage=2, transform=get_pytorch_transforms(224, is_train=False))

stage2_train_loader = DataLoader(stage2_train_ds, batch_size=32, shuffle=True, num_workers=2)
stage2_val_loader = DataLoader(stage2_val_ds, batch_size=32, shuffle=False, num_workers=2)

# Calculate class weights to handle severity imbalance
s2_labels = stage2_train_ds.df['stage_label'].values
class_counts = np.bincount(s2_labels)
class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
class_weights = class_weights / class_weights.sum()
print(f"Stage 2 Class Weights: {class_weights}")

stage2_model = build_stage2_model(backbone='efficientnet_b0', pretrained=True)

print("=== Training Stage 2 Severity Model ===")
s2_history = train_model_pipeline(
    model=stage2_model,
    train_loader=stage2_train_loader,
    val_loader=stage2_val_loader,
    num_epochs=15,
    lr=1e-4,
    class_weights=class_weights,
    save_path='./models/stage2_severity.pth'
)

In [ ]:
# 7. Download Trained Weights to Local Machine
files.download('./models/stage1_binary.pth')
files.download('./models/stage2_severity.pth')
print("Downloaded model weight files!")